# README

- **Author**: RUEI-HSIANG, CHANG（B12303030；AI 協助）
- **Created At**: 2026-10-19
- **Last Modified At**: 2026-10-19

## What does this file do?
四類候選人登記盤點；保留 1,583 筆，不決定最終研究樣本、不刪除資格待查者。
以既有 candidate_key 連接同一來源家族。跨內政部名錄只以正規化縣市＋姓名產生待確認線索，不推定身分或現任。
新擷取學歷與簡歷只列在來源線索，未寫入已核對背景值。
資格來源的搜尋索引指出不符合個案，但本次直接頁面未取得內文，因此全部保留重新核對狀態。

## What does this file take?
- `data/processed/candidates_2026_enriched.csv`；SHA-256 `291a31d7588befded91e8dce0caf7a64cc4cecc927bb4c5fbecd218f6c7281b8`。
- `data/processed/council_candidates_2026_registration.csv`；SHA-256 `e24bba8e93f1f2e58e4d0d180fc74a9cf615930753070dd71cd3e1312ece829b`。
- `data/processed/current_councilors_20261018.csv`；SHA-256 `83ef5ee391fbba602f04ecf316f2aed5cc8f0eb259b5d222e5f1c6b1593d6c67`。
- `data/processed/candidate_campaign_sources.csv`；SHA-256 `682e1fffd9671ae2182548f07a0894b489d8edd1803f4b723106b991115341bd`。
- `data/processed/candidate_campaign_text_blocks.csv`；SHA-256 `3b97686ff977c6f58af3ad84bbd305559d680dcb99fa62a9bb49721c8fa0bc42`。
- `data/processed/meta_candidate_account_worklist_20261018.csv`；SHA-256 `ffdce67367497fbe3776d907fe66a71dde6017cb68832b56df9ce09c6306c9fd`。
- `data/raw/inventory_20261019/capture_manifest.json`；SHA-256 `265f37a902791c15b2b615a9d02ecac4329340d42032cbfd85638d95bb6eb609`。
既有來源說明見 ../processed/candidate_enrichment_documentation.ipynb、../processed/online_20261018_documentation.ipynb。
新增原始頁面見 ../raw/inventory_20261019_documentation.ipynb。

## What does this file output?
- `data/processed/candidate_master_inventory_20261019.csv`；role=collection-inventory；文件 ../processed/candidate_inventory_20261019_documentation.ipynb。
- `data/processed/candidate_field_inventory_20261019.csv`；role=collection-inventory；文件 ../processed/candidate_inventory_20261019_documentation.ipynb。
- `data/processed/candidate_profile_leads_20261019.csv`；role=collection-inventory；文件 ../processed/candidate_inventory_20261019_documentation.ipynb。
- `data/processed/candidate_inventory_summary_20261019.csv`；role=collection-inventory；文件 ../processed/candidate_inventory_20261019_documentation.ipynb。
主表一列一登記者；欄位清單一列一候選人×項目；來源表一列一候選人×名錄線索；摘要一列一公職×欄位×狀態。
輸出皆以字串儲存，缺值空白，未知不填零。下方執行輸出記錄實際筆數與 hash。

修訂：主表party_registration_raw移除PDF排版換行（共8筆），保留政黨文字；上游登記CSV與原始PDF維持原樣。主表每一登記者僅占一個實體文字行。


In [1]:
from pathlib import Path
import hashlib,json,csv,unicodedata
from collections import defaultdict
import pandas as pd
from IPython.display import display
ROOT=next(p for p in [Path.cwd(),*Path.cwd().parents] if (p/'AGENTS.md').exists() and (p/'templates').is_dir())
def sha(p):return hashlib.sha256(p.read_bytes()).hexdigest()
def norm(s):return unicodedata.normalize('NFKC',s).replace('台','臺').replace('温','溫').replace(' ','').replace('　','')
INPUTS={'candidates_2026_enriched.csv': 'data/processed/candidates_2026_enriched.csv', 'council_candidates_2026_registration.csv': 'data/processed/council_candidates_2026_registration.csv', 'current_councilors_20261018.csv': 'data/processed/current_councilors_20261018.csv', 'candidate_campaign_sources.csv': 'data/processed/candidate_campaign_sources.csv', 'candidate_campaign_text_blocks.csv': 'data/processed/candidate_campaign_text_blocks.csv', 'meta_candidate_account_worklist_20261018.csv': 'data/processed/meta_candidate_account_worklist_20261018.csv', 'captures': 'data/raw/inventory_20261019/capture_manifest.json'}
HASHES={'candidates_2026_enriched.csv': '291a31d7588befded91e8dce0caf7a64cc4cecc927bb4c5fbecd218f6c7281b8', 'council_candidates_2026_registration.csv': 'e24bba8e93f1f2e58e4d0d180fc74a9cf615930753070dd71cd3e1312ece829b', 'current_councilors_20261018.csv': '83ef5ee391fbba602f04ecf316f2aed5cc8f0eb259b5d222e5f1c6b1593d6c67', 'candidate_campaign_sources.csv': '682e1fffd9671ae2182548f07a0894b489d8edd1803f4b723106b991115341bd', 'candidate_campaign_text_blocks.csv': '3b97686ff977c6f58af3ad84bbd305559d680dcb99fa62a9bb49721c8fa0bc42', 'meta_candidate_account_worklist_20261018.csv': 'ffdce67367497fbe3776d907fe66a71dde6017cb68832b56df9ce09c6306c9fd', 'captures': '265f37a902791c15b2b615a9d02ecac4329340d42032cbfd85638d95bb6eb609'}
OUTPUTS={'master': 'data/processed/candidate_master_inventory_20261019.csv', 'gaps': 'data/processed/candidate_field_inventory_20261019.csv', 'profiles': 'data/processed/candidate_profile_leads_20261019.csv', 'summary': 'data/processed/candidate_inventory_summary_20261019.csv'}
for key,path in INPUTS.items():assert sha(ROOT/path)==HASHES[key],path
assert len(set(OUTPUTS.values()))==len(OUTPUTS)
assert not set(INPUTS.values()) & set(OUTPUTS.values())
def read(key):return pd.read_csv(ROOT/INPUTS[key],dtype=str,keep_default_na=False).to_dict('records')
m=read('candidates_2026_enriched.csv'); c=read('council_candidates_2026_registration.csv');d=read('current_councilors_20261018.csv')
captures=json.loads((ROOT/INPUTS['captures']).read_text())
for r in captures:
 if r['path']:assert sha(ROOT/r['path'])==r['sha256']
cap={r['source_id']:r for r in captures}


In [2]:
idx=defaultdict(list)
for r in d:idx[(norm(r['county_city']),norm(r['official_name']))].append(r)
accounts=defaultdict(list)
for r in read('meta_candidate_account_worklist_20261018.csv'):accounts[(r['candidate_key'],r['platform'])].append(r)
texts=defaultdict(list)
for r in read('candidate_campaign_text_blocks.csv'):texts[r['candidate_key']].append(r)
sources=defaultdict(list)
for r in read('candidate_campaign_sources.csv'):sources[r['candidate_key']].append(r)
masters=[]; gaps=[]; leads=[]
fieldnames=['sex','birth_date','education','incumbency','first_time_this_office','first_time_any_office','profile','website','Facebook','Instagram','Threads','posts_Facebook','posts_Instagram','posts_Threads','followers','policy_text','visual','political_family','administrative_experience','qualification']
for r in m+c:
 key=r['candidate_key']; mayor=r['office_type'] in ['C1','C2']; office={'C1':'直轄市長','C2':'縣（市）長','縣市議員':'縣（市）議員'}.get(r['office_type'],r['office_type'])
 matches=idx.get((norm(r['county_city']),norm(r['candidate_name'])),[]) if not mayor else []
 master={'candidate_key':key,'candidate_name':r['candidate_name'],'office_type':office,'county_city':r['county_city'],'electoral_district':r.get('electoral_district',''),'party_registration_raw':r.get('party_recommendation_original','').replace('\r','').replace('\n',''),'registration_source':r['source_url'],'registration_source_file':r['source_file'],'registration_source_page':r['source_page'],'qualification_inherited':r['qualification_status'],'qualification_audit':'official_body_recheck_pending','sample_inclusion':'pending_group_rule','directory_lead_count':str(len(matches)),'updated_date':'2026-10-19'}
 for f in fieldnames:
  val='';ev='';status='未查找';note=''
  if mayor:
   mapping={'sex':('sex_2026','sex_source_url'),'birth_date':('birth_date_verified','birth_date_source_url'),'education':('education_profile_as_captured','education_profile_source_url'),'incumbency':('incumbency_2026',''),'first_time_this_office':('first_time_mayor_lifetime',''),'first_time_any_office':('first_time_any_elected_office','prior_any_office_evidence_url'),'profile':('official_profile_url',''),'website':('official_campaign_website','')}
   if f in mapping:
    col,e= mapping[f];val=r.get(col,'');ev=r.get(e,'') if e else ''
    status='已有資料，待來源複核' if val else '既有搜尋後仍缺，待補查'
    if f=='sex':note=r.get('sex_evidence_status','')
    if f=='birth_date':note=r.get('birth_date_evidence_status','')
    if f=='education' and not val:
     val=r.get('education_latest_cec','') or r.get('education_media_supplement','');status='已有資料，待來源複核' if val else status;note='歷史公報或媒體補充，非2026學歷認證'
    if f in ['first_time_this_office','first_time_any_office']:note='0=有過參選證據；空白不表示首次。此處沿用既有表。'
   if f in ['Facebook','Instagram','Threads']:
    acc=accounts.get((key,f),[]);val='; '.join(x['official_urls'] for x in acc if x['official_urls']);ev='; '.join(x['discovered_on'] for x in acc if x['discovered_on']);status='官方連結已保存，帳號內容待核對' if val else '尚未驗證，不代表無帳號'
   if f.startswith('posts_') or f=='followers':status='未取得，不能視為零'
   if f=='policy_text':
    val=str(len(texts[key])) if texts[key] else '';ev='; '.join(sorted({x['source_url'] for x in texts[key]}));status='部分原文已保存，尚未政策編碼' if val else '待補查原文';note='值為HTML段落數，不是獨立政策數；無完整窗口或發布日期'
   if f=='visual':status='未查找';note='舊工作單尚無已驗證截圖路徑'
  if not mayor and f in ['education','incumbency','profile'] and matches:
   status='找到名錄線索，待跨表身分核對';ev='; '.join(x['profile_url'] for x in matches);note='同縣市同姓名命中，未推定為同一人'
  if f=='qualification':val=r['qualification_status'];ev='https://web.cec.gov.tw/central/article/65213';status='官方內文待重新核對';note='保留原始標記；直接下載僅見頁框，不以搜尋摘要直接更新資格或刪人'
  master[f+'_status']=status
  gaps.append({'candidate_key':key,'candidate_name':r['candidate_name'],'office_type':office,'county_city':r['county_city'],'field':f,'status':status,'existing_value':val,'evidence':ev,'note':note,'next_source':{'qualification':'中選會完整審定公告／地方選委會名單','Facebook':'官方人物頁／競選網站交叉連結','Instagram':'官方人物頁／競選網站交叉連結','Threads':'官方人物頁／競選網站交叉連結','policy_text':'選舉公報／候選人政見原文','visual':'候選人官方網站或封面圖'}.get(f,'議會／政府人物頁、歷屆選舉公報；身分與時間核對'),'updated_date':'2026-10-19'})
 for x in matches:
  a=cap.get(x['official_key'],{})
  leads.append({'candidate_key':key,'candidate_name':r['candidate_name'],'county_city':r['county_city'],'directory_key':x['official_key'],'profile_url':x['profile_url'],'match_status':'county_name_match_only_pending_identity','education_raw':a.get('education_raw',''),'experience_raw':a.get('experience_raw',''),'http_status':a.get('http_status',''),'page_contains_name':str(a.get('name_present',False)),'source_file':a.get('path',''),'sha256':a.get('sha256',''),'capture_error':a.get('error',''),'captured_date':'2026-10-19'})
 masters.append(master)
master=pd.DataFrame(masters).fillna(''); gap=pd.DataFrame(gaps).fillna('');lead=pd.DataFrame(leads).fillna('')
summary=gap.groupby(['office_type','field','status'],sort=True).size().reset_index(name='count').astype(str)
outputs={'master':master,'gaps':gap,'profiles':lead,'summary':summary}
assert len(master)==len(m)+len(c)==1583
assert master.candidate_key.is_unique
assert not gap.duplicated(['candidate_key','field']).any()
assert len(gap)==len(master)*len(fieldnames)
assert set(master.office_type)=={'直轄市長','縣（市）長','直轄市議員','縣（市）議員'}
assert set(lead.candidate_key)<=set(master.candidate_key)
display(master.groupby('office_type').size().rename('登記筆數').to_frame())
print('待身分核對來源線索',len(lead),'含學歷原文',sum(lead.education_raw!=''))

# PDF line wrapping is not part of the party name; retain exact source in upstream CSV.
assert not master.party_registration_raw.str.contains(r'[\r\n]',regex=True).any()
example=master.loc[master.candidate_name=='陳玥妗'].iloc[0]
assert example.party_registration_raw=='小民參政歐巴桑聯盟'
assert example.registration_source_page=='42'


,登記筆數
office_type,
直轄市議員,610
直轄市長,23
縣（市）議員,892
縣（市）長,58


待身分核對來源線索 692 含學歷原文 615


In [3]:
verification={}
for key,df in outputs.items():
 p=ROOT/OUTPUTS[key]
 assert p.resolve().is_relative_to((ROOT/'data/processed').resolve())
 # Explicitly scoped outputs only; repeat runs reproduce these inventory files.
 df.to_csv(p,index=False,encoding='utf-8-sig')
 reread=pd.read_csv(p,dtype=str,keep_default_na=False)
 if key=='master':assert len(p.read_text(encoding='utf-8-sig').splitlines())==len(df)+1
 pd.testing.assert_frame_equal(df.astype(str).reset_index(drop=True),reread)
 verification[key]={'path':OUTPUTS[key],'rows':len(reread),'columns':list(reread.columns),'sha256':sha(p),'checks':'schema, exact read-back, declared keys; not analysis-ready'}
 print(key,len(reread),sha(p))
for key,path in INPUTS.items():assert sha(ROOT/path)==HASHES[key]
(ROOT/'data/code/candidate_inventory_20261019_verification.json').write_text(json.dumps(verification,ensure_ascii=False,indent=2))


master 1583 1c5b8e716033949518a71c8972c21b3671de9487e25563d7c37f23b6ceb03474
gaps 31660 89bf3771afe5d0faf2c2e8353ab208a8459dc8b17e336c3da1f4b251473b3f7b
profiles 692 52b723bdeea132df0f5ef9f9ac97594993784ba84edb796c82f0959d63236a3f
summary 108 78dfdb83eac1197585deb6bc82e0e987c7b45fdbfe8c0b7f626ab92809ba242f


2707